# CasTuner Python pipeline — Quickstart on a tiny subset (no FCS required)

This notebook demonstrates the **core logic** of Steps **1a / 1b / 1c** on a *small toy subset*,
without needing the raw `.fcs` files.

Why this exists:
- You can validate the fitting + plotting logic quickly.
- It’s a template you can later swap to real FCS-derived medians.

**What you’ll get:**
- Upregulation half-time (Step 1a)
- Downregulation half-time (Step 1b)
- Hill parameters K, n (Step 1c)

> If you do have real `.fcs` files, run the corresponding scripts directly:
`step_1a_fit_upregulation.py`, `step_1b_fit_downregulation.py`, `step_1c_fit_hill_curves.py`.


In [ ]:
# If you run this notebook inside the project repo:
# %cd /path/to/your/castuner_project

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

rng = np.random.default_rng(7)


## 1) Create a tiny "median table" subset (stands in for gated per-file medians)

In the scripts, the pipeline reads FCS files → gates events → computes channel medians.
Here we fake that final per-file output for **two plasmids** and **two experiments**.


In [ ]:
# Synthetic time points (hours)
t_kd = np.array([0, 1, 2, 4, 8, 12, 24], dtype=float)       # upregulation after withdrawal
t_rev = np.array([0, 1, 2, 4, 8, 12, 24], dtype=float)      # downregulation after addition

plasmids = ["SP411", "SP430A"]
reps = ["R1", "R2"]

# Ground-truth parameters for synthetic generation
t_up_true = {"SP411": 5.0, "SP430A": 8.0}     # hours (half-time)
t_down_true = {"SP411": 6.0, "SP430A": 10.0}  # hours
K_true = {"SP411": 0.35, "SP430A": 0.55}
n_true = {"SP411": 2.0, "SP430A": 1.6}

def exp_rise_to_one(t, t_half):
    # y(t) = 1 - 2^(-t/t_half)
    return 1.0 - 2.0 ** (-t / t_half)

def exp_decay_to_zero(t, t_half):
    # y(t) = 2^(-t/t_half)
    return 2.0 ** (-t / t_half)

# Create toy "normalized BFP" for KD and REV
rows = []
for pl in plasmids:
    for rep in reps:
        for t in t_kd:
            bfp = exp_rise_to_one(t, t_up_true[pl]) + rng.normal(0, 0.03)
            rows.append({"plasmid": pl, "exp": "KD", "rep": rep, "time_h": t, "norm_bfp": np.clip(bfp, 0, 1)})
        for t in t_rev:
            bfp = exp_decay_to_zero(t, t_down_true[pl]) + rng.normal(0, 0.03)
            rows.append({"plasmid": pl, "exp": "Rev", "rep": rep, "time_h": t, "norm_bfp": np.clip(bfp, 0, 1)})

df_tc = pd.DataFrame(rows)
df_tc.head()


## 2) Step 1a — Fit upregulation (KD): exponential rise-to-one, estimate half-time

Model used in the script: rise-to-one with a half-time parameter.
We'll fit per plasmid using pooled replicates.


In [ ]:
def rise_model(t, t_half):
    return 1.0 - 2.0 ** (-t / np.maximum(t_half, 1e-9))

def fit_half_time_rise(sub):
    x = sub["time_h"].to_numpy(float)
    y = sub["norm_bfp"].to_numpy(float)
    p0 = [np.median(x[x>0])] if np.any(x>0) else [5.0]
    popt, pcov = curve_fit(rise_model, x, y, p0=p0, bounds=([1e-3], [1e3]))
    return float(popt[0]), float(np.sqrt(np.diag(pcov))[0])

kd = df_tc.query("exp == 'KD'")
out_up = []
for pl, sub in kd.groupby("plasmid"):
    t_half, se = fit_half_time_rise(sub)
    out_up.append({"plasmid": pl, "t_up_hat": t_half, "se": se})
up_hat = pd.DataFrame(out_up).sort_values("plasmid")
up_hat


In [ ]:
# Plot the fits
fig, ax = plt.subplots()
for pl, sub in kd.groupby("plasmid"):
    ax.scatter(sub["time_h"], sub["norm_bfp"], label=f"{pl} data", alpha=0.7)
    t_half = float(up_hat.loc[up_hat["plasmid"]==pl, "t_up_hat"])
    tt = np.linspace(0, kd["time_h"].max(), 200)
    ax.plot(tt, rise_model(tt, t_half), label=f"{pl} fit (t1/2={t_half:.2f}h)")
ax.set_xlabel("Time (h)")
ax.set_ylabel("Normalized BFP")
ax.set_title("Step 1a — KD upregulation fit (toy subset)")
ax.legend()
plt.show()


## 3) Step 1b — Fit downregulation (Rev): exponential decay, estimate half-time


In [ ]:
def decay_model(t, t_half):
    return 2.0 ** (-t / np.maximum(t_half, 1e-9))

def fit_half_time_decay(sub):
    x = sub["time_h"].to_numpy(float)
    y = sub["norm_bfp"].to_numpy(float)
    p0 = [np.median(x[x>0])] if np.any(x>0) else [6.0]
    popt, pcov = curve_fit(decay_model, x, y, p0=p0, bounds=([1e-3], [1e3]))
    return float(popt[0]), float(np.sqrt(np.diag(pcov))[0])

rev = df_tc.query("exp == 'Rev'")
out_down = []
for pl, sub in rev.groupby("plasmid"):
    t_half, se = fit_half_time_decay(sub)
    out_down.append({"plasmid": pl, "t_down_hat": t_half, "se": se})
down_hat = pd.DataFrame(out_down).sort_values("plasmid")
down_hat


In [ ]:
fig, ax = plt.subplots()
for pl, sub in rev.groupby("plasmid"):
    ax.scatter(sub["time_h"], sub["norm_bfp"], label=f"{pl} data", alpha=0.7)
    t_half = float(down_hat.loc[down_hat["plasmid"]==pl, "t_down_hat"])
    tt = np.linspace(0, rev["time_h"].max(), 200)
    ax.plot(tt, decay_model(tt, t_half), label=f"{pl} fit (t1/2={t_half:.2f}h)")
ax.set_xlabel("Time (h)")
ax.set_ylabel("Normalized BFP")
ax.set_title("Step 1b — Rev downregulation fit (toy subset)")
ax.legend()
plt.show()


## 4) Step 1c — Hill curve fit: fc(mCherry) vs normalized BFP

In the scripts, fold-change is computed relative to NTC and then a Hill curve is fit.
Here we generate toy fold-change values and fit K, n per plasmid.


In [ ]:
def hill_func(x, K, n):
    x = np.asarray(x, float)
    K = np.maximum(float(K), 1e-12)
    n = np.maximum(float(n), 1e-6)
    return (K**n) / (K**n + np.maximum(x, 0.0)**n)

# Synthetic dose points for normalized BFP
x = np.linspace(0.02, 0.98, 12)
rows = []
for pl in plasmids:
    for rep in reps:
        y = hill_func(x, K_true[pl], n_true[pl]) + rng.normal(0, 0.02, size=x.size)
        rows += [{"plasmid": pl, "rep": rep, "norm_bfp": xi, "fc": float(np.clip(yi, 0, 1.2))} for xi, yi in zip(x, y)]
df_hill = pd.DataFrame(rows)

def fit_hill(sub):
    xx = sub["norm_bfp"].to_numpy(float)
    yy = sub["fc"].to_numpy(float)
    p0 = [0.5, 2.0]
    popt, pcov = curve_fit(hill_func, xx, yy, p0=p0, bounds=([1e-6, 0.1], [10.0, 10.0]))
    se = np.sqrt(np.diag(pcov))
    return float(popt[0]), float(popt[1]), float(se[0]), float(se[1])

out = []
for pl, sub in df_hill.groupby("plasmid"):
    Khat, nhat, Kse, nse = fit_hill(sub)
    out.append({"plasmid": pl, "K_hat": Khat, "n_hat": nhat, "K_se": Kse, "n_se": nse})
hill_hat = pd.DataFrame(out).sort_values("plasmid")
hill_hat


In [ ]:
fig, ax = plt.subplots()
for pl, sub in df_hill.groupby("plasmid"):
    ax.scatter(sub["norm_bfp"], sub["fc"], alpha=0.7, label=f"{pl} data")
    Khat = float(hill_hat.loc[hill_hat["plasmid"]==pl, "K_hat"])
    nhat = float(hill_hat.loc[hill_hat["plasmid"]==pl, "n_hat"])
    xx = np.linspace(0, 1, 200)
    ax.plot(xx, hill_func(xx, Khat, nhat), label=f"{pl} fit (K={Khat:.2f}, n={nhat:.2f})")
ax.set_xlabel("Normalized BFP")
ax.set_ylabel("mCherry fold-change (fc)")
ax.set_title("Step 1c — Hill fits (toy subset)")
ax.legend()
plt.show()


## 5) Export toy results (same filenames as the scripts)

These CSVs mimic what the real pipeline produces in `parameters/`.


In [ ]:
out_params = Path("parameters")
out_params.mkdir(exist_ok=True)

up_hat.rename(columns={"t_up_hat":"t_up"}).to_csv(out_params/"half_times_upregulation.csv", index=False)
down_hat.rename(columns={"t_down_hat":"t_down"}).to_csv(out_params/"half_times_downregulation.csv", index=False)
hill_hat.rename(columns={"K_hat":"K","n_hat":"n"}).to_csv(out_params/"Hill_parameters.csv", index=False)

print("Wrote:")
for p in ["half_times_upregulation.csv","half_times_downregulation.csv","Hill_parameters.csv"]:
    print(" -", out_params/p)


## Next: run the full scripts on real data

Once your `.fcs` folders exist, run:

```bash
python step_1a_fit_upregulation.py
python step_1b_fit_downregulation.py
python step_1c_fit_hill_curves.py
```

Those scripts implement the same math, plus the real gating + background subtraction.
